# Control Flow — Polyglot Reference

How execution branches and repeats. Conditional branching, multi-way matching, loops, and the jump statements that escape them.

**Languages:** Java · Scala · Kotlin · JavaScript · TypeScript · Python

This notebook covers:

1. **Conditionals** — `if`/`else`, ternary, the if-is-an-expression divide
2. **Multi-way branching** — `switch` / `match` / `when`, fall-through behavior, expression form, exhaustiveness
3. **Loops, ranges & comprehensions** — for-each, counted, while, ranges, list comprehensions, functional alternatives
4. **Jump statements** — `break`, `continue`, `return`, labeled jumps, Python's `for/else`

Statement-vs-expression semantics for these constructs is covered in `03-operators-expressions.ipynb`. Full pattern matching with destructuring on sealed/case/data classes is in `08-classes-inheritance-matching.ipynb` — this notebook covers only the *control-flow* side of `match`/`when`.

## Conditionals

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| basic `if` | `if (c) { ... } else { ... }` | `if (c) { ... } else { ... }` | `if (c) { ... } else { ... }` | `if (c) { ... } else { ... }` | same | `if c: ...` then `else:` |
| else-if chain | `else if (c2)` | `else if (c2)` | `else if (c2)` | `else if (c2)` | same | `elif c2:` |
| `if` as expression | — | yes | yes | — | — | — *(use `a if c else b`)* |
| ternary | `c ? a : b` | — *(use `if (c) a else b`)* | — *(use `if (c) a else b`)* | `c ? a : b` | same | `a if c else b` |
| guard-clause return | `if (cond) return ...;` | `if (cond) return ...` | `if (cond) return ...` | `if (cond) return ...` | same | `if cond: return ...` |
| parens around condition required | yes | yes | yes | yes | yes | no |
| curly braces required | no *(but recommended)* | no | no | no *(but recommended)* | same | indentation-based |

Scala and Kotlin do not need a separate ternary because `if` already returns a value. Python does not need one because the conditional expression `a if c else b` covers the case — though the postfix word order is unusual. Java, JavaScript, and TypeScript keep the C-style `?:` because their `if` is statement-only.

## Multi-way Branching

This is where the languages diverge most. Pattern-matching capability is a *sliding scale*. Java's old `switch` was constant-only with fall-through; Java 14 added the switch expression with arrow syntax; Java 21 added type and record patterns. Kotlin's `when` has always been an expression. Python added `match`-case in 3.10 as full structural matching. Scala's `match` is the most powerful.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| keyword | `switch` | `match` | `when` | `switch` | `switch` | `match` *(3.10+)* |
| basic case syntax | `case 1: ...; break;` | `case 1 => ...` | `1 -> ...` | `case 1: ...; break;` | same | `case 1: ...` |
| arrow syntax *(no break)* | switch-expr *(14+)* | always | always | — | — | always |
| produces a value | switch-expr *(14+)* | yes | yes | — | — | — *(statement only)* |
| fall-through | yes *(by default)* | — | — | yes *(by default)* | yes | — |
| match by type | `case Foo f` *(21+)* | `case f: Foo =>` | `is Foo` | — | — | `case Foo():` |
| guards | `case Foo f when f.x > 0` *(21+)* | `case x if x > 0 =>` | — *(workaround in `if`)* | — | — | `case x if x > 0:` |
| default case | `default:` | `case _ =>` | `else ->` | `default:` | same | `case _:` |
| exhaustiveness check | sealed switch *(21+)* | sealed match | sealed `when` *(when used as expr)* | — | — | structural narrowing *(3.10+)* |

**Kotlin `when` without a subject** acts as an if-else-chain replacement:
```
when {
  x > 0 -> "positive"
  x < 0 -> "negative"
  else  -> "zero"
}
```

## Loops, Ranges & Comprehensions

### Loop forms

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| C-style `for(i=0;i<n;i++)` | yes | — | — | yes | yes | — |
| for-each | `for (T x : coll)` | `for (x <- coll)` | `for (x in coll)` | `for (const x of coll)` | same | `for x in coll:` |
| `for...in` vs `for...of` | n/a | n/a | n/a | `for...in` iterates **keys**; `for...of` iterates **values** | same | n/a |
| while | `while (c) { ... }` | `while (c) { ... }` | `while (c) { ... }` | `while (c) { ... }` | same | `while c: ...` |
| do-while | `do { ... } while (c);` | — *(removed in Scala 3)* | `do { ... } while (c)` | `do { ... } while (c);` | same | — |
| infinite loop | `while (true)` / `for (;;)` | `while (true)` | `while (true)` | `while (true)` / `for (;;)` | same | `while True:` |
| iterate with index | manual counter | `for ((x, i) <- c.zipWithIndex)` | `for ((i, x) in c.withIndex())` | `for (const [i, x] of c.entries())` | same | `for i, x in enumerate(c):` |
| iterate map / dict | `for (var e : map.entrySet())` | `for ((k, v) <- map)` | `for ((k, v) in map)` | `for (const [k, v] of Object.entries(o))` | same | `for k, v in d.items():` |

### Ranges

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| exclusive end | `IntStream.range(0, n)` | `0 until n` | `0 until n` | manual `for` | manual `for` | `range(n)` |
| inclusive end | `IntStream.rangeClosed(0, n)` | `0 to n` | `0..n` | manual `for` | manual `for` | `range(n+1)` |
| with step | `IntStream.iterate(...)` | `0 until n by 2` | `0 until n step 2` | manual `for` | manual `for` | `range(0, n, 2)` |
| descending | reverse Stream | `n to 0 by -1` | `n downTo 0` | manual `for` | manual `for` | `range(n, -1, -1)` |

### Comprehensions & functional alternatives

Python and Scala have *syntactic* comprehensions. The other four use chained functional methods to express the same idea.

| Operation | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| filter + map | `s.filter(p).map(f)` *(Stream)* | `for { x <- xs if p(x) } yield f(x)` | `xs.filter(p).map(f)` | `xs.filter(p).map(f)` | same | `[f(x) for x in xs if p(x)]` |
| flatten | `flatMap` | nested for-yield | `flatMap` | `flatMap` | same | nested `for` in comprehension |
| reduce / fold | `reduce(0, Integer::sum)` | `xs.foldLeft(0)(_+_)` | `xs.fold(0) { a, x -> a+x }` | `xs.reduce((a, x) => a+x, 0)` | same | `functools.reduce` or `sum(xs)` |
| find first | `findFirst` | `xs.find(p)` | `xs.find(p)` | `xs.find(p)` | same | `next(x for x in xs if p(x))` |
| dict / map comprehension | `Collectors.toMap` | `xs.map(x => x -> f(x)).toMap` | `xs.associateWith { f(it) }` | `Object.fromEntries(...)` | same | `{x: f(x) for x in xs}` |
| any / all | `anyMatch` / `allMatch` | `xs.exists` / `xs.forall` | `any` / `all` | `some` / `every` | same | `any(...)` / `all(...)` |

## Jump Statements

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| `break` from loop | yes | — *(use `scala.util.control.Breaks`)* | yes | yes | yes | yes |
| `continue` | yes | — *(use guard in for-comprehension)* | yes | yes | yes | yes |
| `return` from function | yes | yes | yes | yes | yes | yes |
| labeled `break` | `break label;` | — | `break@label` | `break label;` | same | — |
| labeled `continue` | `continue label;` | — | `continue@label` | `continue label;` | same | — |
| non-local return from lambda | — *(use `return` in named method)* | yes *(throws via library)* | `return@outerFunction` | — | — | — |
| `for/else`, `while/else` | — | — | — | — | — | yes — `else` runs if loop did **not** `break` |

Scala's omission of `break`/`continue` is deliberate. The expressive substitutes are filter clauses inside for-comprehensions (`if p`), recursive helpers, and `xs.find` / `takeWhile` / `dropWhile` for early exit. The `scala.util.control.Breaks.breakable { ... break() }` library escape hatch exists but is rarely idiomatic.

## Notes — when a cell isn't enough

**Java old switch vs switch expression.** The traditional `switch (x) { case 1: ...; break; ... }` is a *statement* with fall-through-by-default — forgetting `break` is the most common bug. Java 14+ added the *switch expression* with arrow syntax (`case 1 -> foo()`). No fall-through, returns a value, can be exhaustive over sealed types. Java 21 layered on type and record patterns. Use the arrow form in new code.

**JavaScript / TypeScript `switch` is C-style.** Each `case` falls through to the next unless you `break` or `return`. This is the single most common JavaScript control-flow bug. There is no expression form of `switch` — `?:` is the only multi-way branch that produces a value. ESLint has rules for fall-through detection; turn them on.

**Kotlin's `when` is universally useful.** With a subject: `when (x) { 1 -> ...; in 1..10 -> ...; is String -> ...; else -> ... }`. Without a subject: `when { x > 0 -> "pos"; x < 0 -> "neg"; else -> "zero" }` — replaces if-else chains. Always an expression. Exhaustiveness is checked when `when` is used as an expression over a sealed type.

**Python `match` is structural, not C-style switch.** Despite the surface syntax, Python's `match` (3.10+) is pattern matching with destructuring, not a `switch` lookalike. `case 1` matches the literal `1`; `case [x, y]` destructures a 2-length sequence; `case Point(x=0)` matches a `Point` with `x=0`; `case {"name": name}` matches a dict with a `name` key. The control-flow side is covered here; destructuring power is covered in `08-classes-inheritance-matching`.

**Scala has no `break` / `continue`.** By design — they were considered uncomposable. Substitutes: guard expressions inside for-comprehensions (`for { x <- xs if p(x) } yield ...`), recursive helpers with tail-call optimization, `xs.find` / `takeWhile` / `dropWhile` / `span`. For the rare unavoidable case, `scala.util.control.Breaks.breakable { ... break() }` exists. If you reach for it more than once a quarter, you're probably structuring loops wrong.

**Python `for/else` and `while/else`.** Python loops have an `else` clause that runs *only if* the loop completed without `break`. Useful idiom for search-and-report: write the success path inside the loop with `break`, write the not-found path inside `else`. Famously confusing the first time — the keyword `else` is misleading; mentally read it as `nobreak`. No other language here has this.

**JavaScript `for...in` vs `for...of`.** `for (const k in obj)` iterates **keys** (including inherited ones, sometimes — use `hasOwn` to filter). `for (const v of iter)` iterates **values** (and requires an iterable, including arrays, maps, sets, generators). Modern JavaScript almost always wants `for...of`. The `in` form is a legacy of when objects-as-dictionaries was the only collection type.

**Comprehensions vs functional chains.** Python's `[f(x) for x in xs if p(x)]` and Scala's `for { x <- xs if p(x) } yield f(x)` are *syntactic sugar* for `xs.filter(p).map(f)`. Java Streams, Kotlin sequence ops, and JavaScript array methods express the same idea functionally. Comprehensions read more naturally for nested iteration (`for x in xs for y in ys`); functional chains compose better and short-circuit more easily on infinite/lazy sources.

**Range syntax divergence.** Scala uses words: `0 to n` (inclusive), `0 until n` (exclusive). Kotlin uses operators: `0..n` (inclusive), `0 until n` (exclusive), `n downTo 0` (descending). Python uses the function form: `range(n)` (exclusive), `range(start, stop, step)`. Java has no native range — `IntStream.range(0, n)` fills the gap. JavaScript has no range at all — `Array.from({length: n}, (_, i) => i)` is the closest idiom.

**Iterate with index.** Python's `enumerate` is the cleanest (`for i, x in enumerate(coll):`). Kotlin's `withIndex()` is also clean. Scala's `zipWithIndex` is verbose but composable. Java's stream approach requires `IntStream.range(0, list.size())` and manual indexing. JavaScript's `array.entries()` returns `[index, value]` pairs and requires array destructuring in the loop variable.

**Non-local return on the JVM.** Scala and Kotlin allow `return` from inside a lambda to exit the *enclosing function*, not the lambda itself. Scala implements this via a thrown exception (a real performance cost in hot paths). Kotlin requires `return@label` or `return@functionName` syntax to make the target explicit. Java forbids non-local return outright — `return` always exits the immediately enclosing method.

**Do-while removed in Scala 3.** Scala 3 dropped `do-while` from the language. The replacement is `while ({ body; condition }) ()` — a `while` with the body and condition swapped. Rarely needed; do-while is a niche form even in languages that have it.

**Pattern matching is a sliding scale.** Java 0–13: constant-only switch. Java 14+: switch expression. Java 21+: type and record patterns plus guards. JavaScript / TypeScript: no pattern matching, only constant switch. Kotlin: type and range patterns in `when`. Python 3.10+: full structural matching. Scala 2/3: full structural matching with extractors. When you're cross-language, knowing where each language sits on the ladder is worth more than memorizing case syntax.